# New Scheduled Job

This notebook performs a system health check and logs results to `data_out/`.


In [5]:
import datetime
import json
import os
from pathlib import Path

# --- Config ---
OUTPUT_DIR = Path("data_out/scheduled_job")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATUS_FILE = OUTPUT_DIR / "status.json"
LOG_FILE = OUTPUT_DIR / "job.log"

run_time = datetime.datetime.utcnow().isoformat()
print(f"Job started at {run_time}")

Job started at 2026-07-07T04:21:12.737213


/tmp/ipykernel_33285/1797822990.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_time = datetime.datetime.utcnow().isoformat()


In [6]:
# --- Health Checks ---
checks = {}

# Python version
import sys

checks["python_version"] = sys.version

# Check key directories exist
for d in ["datasets", "data_out", "scripts", "shared", "apps"]:
    checks[f"dir_{d}"] = os.path.isdir(d)

# Check optional env vars
for var in ["AZURE_OPENAI_API_KEY", "OPENAI_API_KEY", "QAI_DB_CONN", "OLLAMA_BASE_URL"]:
    checks[f"env_{var}"] = var in os.environ

# Disk usage
import shutil

total, used, free = shutil.disk_usage("/")
checks["disk_free_gb"] = round(free / (1024**3), 2)
checks["disk_used_pct"] = round(used / total * 100, 1)

print(json.dumps(checks, indent=2))

{
  "python_version": "3.14.0 (main, Oct  8 2025, 21:26:42) [GCC 14.2.0]",
  "dir_datasets": true,
  "dir_data_out": true,
  "dir_scripts": true,
  "dir_shared": true,
  "dir_apps": true,
  "env_AZURE_OPENAI_API_KEY": false,
  "env_OPENAI_API_KEY": false,
  "env_QAI_DB_CONN": false,
  "env_OLLAMA_BASE_URL": true,
  "disk_free_gb": 827.63,
  "disk_used_pct": 12.7
}


In [7]:
# --- Write Status File ---
status = {
    "last_run": run_time,
    "checks": checks,
    "status": "ok" if all(v for k, v in checks.items() if k.startswith("dir_")) else "warn",
}

STATUS_FILE.write_text(json.dumps(status, indent=2))

# Append to log
with LOG_FILE.open("a") as f:
    f.write(f"{run_time} | status={status['status']} | disk_free={checks['disk_free_gb']}GB\n")

print(f"Status: {status['status']}")
print(f"Results written to {STATUS_FILE}")

Status: ok
Results written to data_out/scheduled_job/status.json


## Summary

- Checks all key directories (`datasets/`, `data_out/`, `scripts/`, `shared/`, `apps/`)
- Detects optional env vars (Azure OpenAI, OpenAI, DB, Ollama)
- Reports disk usage
- Writes machine-readable `status.json` and appends to `job.log` in `data_out/scheduled_job/`


In [8]:
# --- Post-Run Validation & Exit Marker ---
status_data = json.loads(STATUS_FILE.read_text())
required = ["last_run", "checks", "status"]
missing = [k for k in required if k not in status_data]
if missing:
    raise RuntimeError(f"Invalid status file; missing keys: {missing}")

ok_dirs = sum(1 for k, v in status_data["checks"].items() if k.startswith("dir_") and v)
all_dirs = sum(1 for k in status_data["checks"].keys() if k.startswith("dir_"))
print(
    f"Health summary: status={status_data['status']} dirs={ok_dirs}/{all_dirs} disk_free={status_data['checks'].get('disk_free_gb')}GB"
)

# Scheduler-friendly marker (parseable in logs)
exit_code = 0 if status_data["status"] == "ok" else 1
print(f"JOB_EXIT_CODE={exit_code}")

Health summary: status=ok dirs=5/5 disk_free=827.63GB
JOB_EXIT_CODE=0
